# Time series Notebook
The aim of this notebook is to compare the timeseries of topic inside/between clusters.

## Load libraries

In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
import pipeline.src.python.config as cfg
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', None)
from bertopic import BERTopic

/home/banfi/.uve/cuda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data and create the timeseries dataset

In [3]:
cluster_df = pd.read_parquet('community_detection_results.parquet')

In [4]:
cluster_df

,Topic Label,Cluster,Model,Cluster Label
0,Tuberculosis Resistance and Control,24,scopus,Tuberculosis Management Strategies
1,Tuberculosis Treatment Challenges,24,the_guardian,Tuberculosis Management Strategies
2,Zoonotic Disease Surveillance,0,scopus,Infectious Disease and Surveillance
3,Disease Spread and Environmental Impact,0,the_guardian,Infectious Disease and Surveillance
4,Infection Control and Prevention,10,scopus,Hospital Infection Management
...,...,...,...,...
244,Immune Response in Viral Infections,11,science_news,Long-Term Health Impacts of COVID19
245,Vitamin Status and Supplementation in COVID-19 Patients,11,scopus,Long-Term Health Impacts of COVID19
246,Covid-19 Severity Biomarkers,11,scopus,Long-Term Health Impacts of COVID19
247,Cov-2 Serological Testing,11,scopus,Long-Term Health Impacts of COVID19


In [5]:
model_list = ['scopus','science_news','the_guardian']

In [6]:
timeseries_df = pd.DataFrame()

In [7]:
for model in model_list:
    
    cfg_dict = cfg.MAGAZINE_CONFIG[model]
    bertopic_model = BERTopic.load(cfg_dict['REFERENCE_MODEL'])

    model_data = np.load(cfg_dict['OUTPUT_PATH'],allow_pickle=True)

    dataset = pd.read_parquet(cfg_dict['DATASET_PATH'])
    
    tmp_df = bertopic_model.get_document_info(model_data['text'])['CustomName'].reset_index()
    tmp_df = tmp_df.drop(columns='index')

    tmp_df['id'] =  list(model_data['id'])
    tmp_df = tmp_df.rename(columns={'CustomName':'Topic Label'})
    
    tmp_df = tmp_df.merge(cluster_df,on='Topic Label')

    columns_to_mantain = ['id','publicationDate']
    columns_to_drop = [column for column in dataset.columns if column not in columns_to_mantain ]

    tmp_df = tmp_df.merge(dataset.drop(columns=columns_to_drop),on='id')

    timeseries_df = pd.concat([timeseries_df,tmp_df])


2026-02-19 08:56:02,908 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.
2026-02-19 08:56:12,430 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.
2026-02-19 08:56:14,118 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


## Select and plot clusters

In [8]:
cluster_df[['Cluster','Cluster Label']].value_counts().reset_index().sort_values(by='Cluster')

,Cluster,Cluster Label,count
2,0,Infectious Disease and Surveillance,20
0,1,Coronavirus Research and Response,25
1,2,AntimicrobialResistance Challenges,22
3,3,Vaccine Research and Public Health,19
6,4,Influenenza Viral Dynamics,10
5,5,Zoonotic Disease Origins and Spread,12
7,6,Antiviral Treatments and Therapies,8
10,7,Pandemic and Health Crises,7
9,8,HIV Research and Public Health,7
12,9,Ebola Outbreak Management,5


In [9]:
#Covid
#cluster_list = [1,7,11,13,17,19,25,30,36,37,38]
# HIV
#cluster_list = [8,18,32]
# Influenza
#cluster_list = [4,29]
#Ebola
#cluster_list= [9]
#Malaria
#cluster_list = [12]
# Mpox
#cluster_list = [28]
# HAI and AMR
cluster_list = [10]

In [10]:
timeseries_df_analysis = timeseries_df[ timeseries_df['Cluster'].isin(cluster_list) ]

In [11]:
if len(cluster_list) > 1:
    # Settare un identificativo per Cluster
    if 'Cluster Label' in timeseries_df_analysis.columns:
        timeseries_df_analysis['Graph_label'] = timeseries_df_analysis['Cluster Label']
    else:
        timeseries_df_analysis['Graph_label'] = timeseries_df_analysis['Cluster']
else:
    # Settare un identificativo per Topic Label
    timeseries_df_analysis['Graph_label'] = timeseries_df_analysis['Topic Label']


In [13]:
import polars as pl

timeseries_df_analysis_polars = pl.from_pandas(timeseries_df_analysis)

In [14]:
#period = '1mo'
period = '1y'

In [15]:
timeseries_df_analysis_polars = timeseries_df_analysis_polars.with_columns(
    pl.col('publicationDate').dt.truncate(period).alias('timestamp'),
)

In [ ]:
partial_timeseries_df_analysis_polars = timeseries_df_analysis_polars.group_by(['Graph_label','timestamp']).agg(
        pl.len().alias('occurrence')
    ).sort(by=['timestamp','Graph_label'],descending=False)

In [19]:
dates = pl.date_range(
    start= partial_timeseries_df_analysis_polars.select(pl.col("timestamp").min()).item(),
    end=  partial_timeseries_df_analysis_polars.select(pl.col("timestamp").max()).item(),
    interval= period,
    eager = True
)

In [33]:
topics = ( 
    partial_timeseries_df_analysis_polars.select(
        pl.col('Graph_label').unique().sort()).to_series()
)

In [34]:
grid = (
    pl.DataFrame({'Graph_label':topics}).join(pl.DataFrame({'timestamp':dates}),how='cross')
)

In [35]:
partial_timeseries_df_analysis_polars = partial_timeseries_df_analysis_polars.with_columns(
        pl.col("timestamp").dt.date().alias("timestamp")
)

In [36]:
complete_timeseries_df_analysis_polars  = (
    grid.join(partial_timeseries_df_analysis_polars,on=['Graph_label','timestamp'],how='left')
    .with_columns(pl.col('occurrence').fill_null(0))
    .sort(by=['timestamp','Graph_label'],descending=False)
)

In [37]:
complete_timeseries_df_analysis = complete_timeseries_df_analysis_polars.to_pandas()

In [39]:
import plotly.express as px

if len(cluster_list) > 1:
    legend_title = "Cluster"
    graph_title = "Cluster time series"
else:
    legend_title = "Topics"
    graph_title= "Topics time series"

fig = px.line(
    complete_timeseries_df_analysis,
    x="timestamp",
    y="occurrence",
    color="Graph_label",
    markers=True,
    title=graph_title
)

fig.update_layout(
    # Titolo
    title={
        'text': graph_title,
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20, 'family': 'Arial, sans-serif'}
    },
    
    # Assi
    xaxis_title="Time",
    yaxis_title="# Articles",
    xaxis=dict(
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray',
        showline=True,
        linewidth=2,
        linecolor='black'
    ),
    yaxis=dict(
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray',
        showline=True,
        linewidth=2,
        linecolor='black'
    ),    
    legend=dict(
        title=legend_title,
        orientation="h",  
        yanchor="top",
        y=-0.15, 
        xanchor="center",
        x=0.5,  
        bgcolor="rgba(255, 255, 255, 0.8)",
        bordercolor="Black",
        borderwidth=1
    ),
    
    
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=1000,
    height=600,
    
    
    margin=dict(l=80, r=80, t=80, b=120),
    
    
    hovermode='x unified'
)


fig.update_traces(
    line=dict(width=2.5),
    marker=dict(size=6)
)

fig.show()

### New graph

In [236]:
model_articles_per_year = pd.read_parquet('model_articles_per_year.parquet')

In [237]:
cluster_list = [20]

In [238]:
timeseries_df_analysis = timeseries_df[ (timeseries_df['Cluster'].isin(cluster_list)) & (timeseries_df['publicationDate'].dt.year <= 2025)  ]

In [239]:
timeseries_df_analysis['Model'] = timeseries_df_analysis['Model'].apply( lambda x: x.title().replace('_',' '))

In [240]:
if len(cluster_list) > 1:
    # Settare un identificativo per Cluster
    if 'Cluster Label' in timeseries_df_analysis.columns:
        timeseries_df_analysis['Graph_label'] = timeseries_df_analysis['Cluster Label']
    else:
        timeseries_df_analysis['Graph_label'] = timeseries_df_analysis['Cluster']
else:
    # Settare un identificativo per Topic Label
    timeseries_df_analysis['Graph_label'] = timeseries_df_analysis['Topic Label']

In [241]:
period = '1y'

In [242]:
timeseries_df_analysis_polars = pl.from_pandas(timeseries_df_analysis).with_columns(
    pl.col('publicationDate').dt.truncate(period).alias('timestamp'),
)

In [243]:
partial_timeseries_df_analysis_polars = timeseries_df_analysis_polars.group_by(['Graph_label','Model','timestamp']).agg(
        pl.len().alias('occurrence')
    ).sort(by=['timestamp','Model','Graph_label'],descending=False)

In [244]:
dates = pl.date_range(
    start= partial_timeseries_df_analysis_polars.select(pl.col("timestamp").min()).item(),
    end=  partial_timeseries_df_analysis_polars.select(pl.col("timestamp").max()).item(),
    interval= period,
    eager = True
)

In [245]:
topics = ( 
    partial_timeseries_df_analysis_polars.select(['Graph_label','Model']).unique()
)

In [246]:
grid = (
    topics.join(pl.DataFrame({'timestamp':dates}),how='cross')
)

In [247]:
partial_timeseries_df_analysis_polars = partial_timeseries_df_analysis_polars.with_columns(
        pl.col("timestamp").dt.date().alias("timestamp")
)

In [248]:
complete_timeseries_df_analysis_polars  = (
    grid.join(partial_timeseries_df_analysis_polars,on=['Graph_label','Model','timestamp'],how='left')
    .with_columns(pl.col('occurrence').fill_null(0))
    .sort(by=['timestamp','Model','Graph_label'],descending=False)
)

In [249]:
complete_timeseries_df_analysis = complete_timeseries_df_analysis_polars.to_pandas()

In [252]:
complete_timeseries_df_analysis = complete_timeseries_df_analysis.merge(model_articles_per_year,on=['Model','timestamp'])

In [253]:
complete_timeseries_df_analysis

,Graph_label,Model,timestamp,occurrence,Articles
0,Coronavirus Outbreak Spread,Science News,2001-01-01,0,1128
1,COVID-19 Exposure and Infection in Healthcare Workers,Scopus,2001-01-01,0,1241646
2,COVID-19 Mortality and Severe Disease Outcomes,Scopus,2001-01-01,0,1241646
3,Coronavirus Disease Outbreak,Scopus,2001-01-01,0,1241646
4,Covid-19 Chest Imaging Findings,Scopus,2001-01-01,0,1241646
...,...,...,...,...,...
195,Coronavirus Disease Outbreak,Scopus,2025-01-01,3,4304715
196,Covid-19 Chest Imaging Findings,Scopus,2025-01-01,2,4304715
197,HFMD Outbreak and Surveillance,Scopus,2025-01-01,16,4304715
198,China's SARS and COVID Response Strategy,The Guardian,2025-01-01,2,72085


In [275]:
complete_timeseries_df_analysis['frequency_per_year'] = (complete_timeseries_df_analysis['occurrence'] / complete_timeseries_df_analysis['Articles']) * 100_000

In [276]:
complete_timeseries_df_analysis

,Graph_label,Model,timestamp,occurrence,Articles,frequency_per_year
0,Coronavirus Outbreak Spread,Science News,2001-01-01,0,1128,0.000000
1,COVID-19 Exposure and Infection in Healthcare Workers,Scopus,2001-01-01,0,1241646,0.000000
2,COVID-19 Mortality and Severe Disease Outcomes,Scopus,2001-01-01,0,1241646,0.000000
3,Coronavirus Disease Outbreak,Scopus,2001-01-01,0,1241646,0.000000
4,Covid-19 Chest Imaging Findings,Scopus,2001-01-01,0,1241646,0.000000
...,...,...,...,...,...,...
195,Coronavirus Disease Outbreak,Scopus,2025-01-01,3,4304715,0.069691
196,Covid-19 Chest Imaging Findings,Scopus,2025-01-01,2,4304715,0.046461
197,HFMD Outbreak and Surveillance,Scopus,2025-01-01,16,4304715,0.371685
198,China's SARS and COVID Response Strategy,The Guardian,2025-01-01,2,72085,2.774502


In [277]:
how = 'frequency_per_year'
#how = 'occurrence'

In [279]:
import plotly.express as px
import pandas as pd

models  = list(pd.unique(complete_timeseries_df_analysis["Model"]))

if len(cluster_list) > 1:
    legend_title = "Cluster"
    graph_title = "Cluster time series"
else:
    legend_title = "Topics"
    graph_title= "Topics time series"

fig = px.line(
    complete_timeseries_df_analysis,
    x="timestamp",
    y=how,
    color="Graph_label",
    markers=True,
    line_group="Graph_label",
    facet_row="Model",
    title=graph_title,
    facet_row_spacing=0.12
)


fig.update_layout(
    title={
        'text': graph_title,
        'x': 0.5,
        'y' : 0.98,
        'xanchor': 'center',
        'font': {'size': 20, 'family': 'Arial, sans-serif'}
    },
    legend=dict(
        title=legend_title,
        orientation="h",
        yanchor="top",
        y=-0.15,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255, 255, 255, 0.8)",
        bordercolor="Black",
        borderwidth=1
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=1000,
    height=700,
    margin=dict(l=80, r=80, t=80, b=120),
    hovermode='x unified'
)


for r in range(1, len(models) + 1):
    fig.update_xaxes(
        title_text="Time" if r == 1 else None,  
        row=r,
        col=1,
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray',
        showline=True,
        linewidth=2,
        linecolor='black',
        showticklabels=True,   
        ticks="outside"
    )

for r in range(1, len(models) + 1):
    fig.update_yaxes(
        title_text="# Articles" if r == 2 and how=='occurrence' else '# Articles per 100k' if r == 2 and how=='frequency_per_year' else  None,
        row=r,
        col=1,
        showgrid=True,
        gridwidth=1,
        gridcolor='lightgray',
        showline=True,
        linewidth=2,
        linecolor='black',
    )

for ann in fig.layout.annotations:
    if ann.text.startswith("Model="):
        ann.text = ann.text.replace("Model=", "") 
        ann.x = 0.5                                
        ann.xanchor = "center"
        ann.y += 0.15                              
        ann.yanchor = "bottom"
        ann.font = dict(size=15, family="Arial, sans-serif")
        ann.textangle = 0 


fig.update_yaxes(matches=None)


fig.update_traces(
    line=dict(width=2.5),
    marker=dict(size=6)
)

fig.show()
